# 01 — Análisis exploratorio

Antes de tocar un solo modelo conviene mirar el dataset. El objetivo del notebook no es ser exhaustivo —no nos interesan estadísticas que no nos vayan a llevar a una decisión— sino contestar a unas cuantas preguntas concretas:

1. ¿Cómo está repartido el dataset entre splits, categorías y dificultades?
2. ¿Está equilibrado por clase o vamos a tener que compensar el desbalance en el entrenamiento?
3. ¿Qué pinta tienen las imágenes? ¿Se distinguen las clases a simple vista?
4. ¿Los píxeles aportan información discriminante o nuestras imágenes son demasiado parecidas en intensidad?
5. ¿Cómo se proyectan las clases en un espacio reducido (PCA, t-SNE)? ¿Se separan las categorías Inorgánica vs. Orgánica?
6. ¿Qué pares de clases nos vamos a confundir con más probabilidad?

Cada apartado contiene la respuesta a una de estas preguntas, y al final cerramos con las decisiones concretas que se derivan del análisis y que aplicamos en los notebooks 02 a 06.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch, numpy as np, random
torch.manual_seed(42); np.random.seed(42); random.seed(42)

METADATA_PATH = ROOT / 'data' / 'metadata.csv'
RAW_DATA_DIR  = ROOT / 'data' / 'raw'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
df = pd.read_csv(METADATA_PATH)
print(f'Total imágenes: {len(df)}, clases: {df["compound_id"].nunique()}')

## Volumen y reparto

Lo primero es ver cuántas imágenes tenemos y cómo se reparten. Esperamos un 70/15/15 train/val/test prácticamente exacto (lo configuramos así en el generador) y un reparto inorgánica/orgánica de aproximadamente 55/45, dado el tamaño relativo de los dos catálogos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['split'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribución por split'); axes[0].set_ylabel('# imágenes')
df['category'].value_counts().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Distribución por categoría')
df['difficulty'].value_counts().plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Distribución por dificultad')
for ax in axes: ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

print(df.groupby(['split', 'category']).size().unstack(fill_value=0))

## Desbalance de clases

Aunque hayamos generado exactamente 300 imágenes por compuesto, el desbalance puede aparecer por dos vías: o bien porque el usuario filtre por subcategoría en la demo (notebook 06), o bien porque alguna subcategoría tenga muchos más compuestos que otra. Como medida agregada usamos el **índice de Gini** sobre el vector de cuentas por clase. Un Gini próximo a 0 indica reparto perfectamente uniforme y un Gini próximo a 1 indica que casi todas las imágenes son de una única clase.

In [ ]:
counts = df['compound_id'].value_counts().sort_values()

def gini(arr):
    arr = np.sort(np.asarray(arr, dtype=float))
    n = len(arr)
    if arr.sum() == 0:
        return 0.0
    return (2 * np.sum((np.arange(1, n+1)) * arr) / (n * arr.sum())) - (n + 1) / n

g = gini(counts.values)
print(f'Índice de Gini = {g:.3f} (0 = perfectamente equilibrado, 1 = totalmente desigual)')
print(f'Min/clase: {counts.min()}   Max/clase: {counts.max()}   Mediana: {counts.median():.0f}')

fig, ax = plt.subplots(figsize=(12, max(4, len(counts)*0.12)))
counts.plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(counts.mean(), color='red', linestyle='--', label=f'Media={counts.mean():.0f}')
ax.set_xlabel('# imágenes'); ax.set_title('Imágenes por clase')
ax.legend(); plt.tight_layout(); plt.show()

El Gini sale prácticamente 0, lo cual es coherente con la decisión del notebook 00 de generar el mismo número de imágenes por clase. Aun así, dejamos activado `WeightedRandomSampler` en `src/dataset.py` por dos motivos: añade robustez si en el futuro alguien filtra el dataset, y el coste computacional adicional es despreciable.

## Inspección visual

Pintamos una rejilla con un ejemplo por cada uno de 20 compuestos elegidos al azar. La cantidad de información que contiene esta figura es alta: se aprecian los dos modos de render (estructura 2D vs texto), el efecto de la aumentación (las imágenes están rotadas y con ruido), y empezamos a intuir qué pares de compuestos van a ser difíciles. Por ejemplo, dos alcanos consecutivos (`pentano` y `hexano`) sólo se diferencian en un carbono y la diferencia visual en 224×224 después de aplicar deformación elástica es muy sutil.

In [ ]:
sample_ids = df['compound_id'].drop_duplicates().sample(min(20, df['compound_id'].nunique()), random_state=0)
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
for ax, cid in zip(axes.flat, sample_ids):
    r = df[df['compound_id'] == cid].iloc[0]
    img = Image.open(ROOT / r['filepath'])
    ax.imshow(img); ax.axis('off')
    ax.set_title(f"{r['formula']}", fontsize=8)
for ax in axes.flat[len(sample_ids):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## Distribución de intensidad

Las imágenes son negras sobre fondo blanco, así que la inmensa mayoría de los píxeles serán claros y un puñado oscuros (los trazos). Esto debería verse como un histograma muy sesgado a la derecha. Como sanity check, comparamos la intensidad media entre las dos categorías para detectar posibles fugas de información obvias (si una categoría tuviese el fondo gris en vez de blanco, lo veríamos aquí).

In [ ]:
sample = df.sample(min(500, len(df)), random_state=42)
all_pixels = []
for fp in sample['filepath']:
    arr = np.array(Image.open(ROOT / fp).convert('L'))
    all_pixels.append(arr.flatten())
all_pixels = np.concatenate(all_pixels)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(all_pixels, bins=50, color='steelblue')
axes[0].set_title('Histograma global de intensidades (gris)')
axes[0].set_xlabel('Intensidad'); axes[0].set_ylabel('Frecuencia')

per_cat = sample.copy()
per_cat['mean'] = [np.array(Image.open(ROOT / fp).convert('L')).mean() for fp in per_cat['filepath']]
per_cat.boxplot(column='mean', by='category', ax=axes[1])
axes[1].set_title('Intensidad media por categoría'); axes[1].set_ylabel('Media de píxel')
plt.suptitle('')
plt.tight_layout(); plt.show()

## Reducción de dimensionalidad

Aplicamos PCA y t-SNE sobre una muestra del dataset (reducida a 32×32 en gris para hacerlo manejable) y proyectamos a 2D. Es una manera rápida de ver si las dos grandes categorías son separables ya en el espacio crudo de píxeles, antes incluso de pasar por un modelo.

Lo más interesante suele ser el t-SNE: si vemos dos grandes manchas correspondientes a inorgánica/orgánica, sabemos que un modelo lineal sobre píxeles ya debería aprender una frontera razonable. Si en cambio están totalmente mezcladas, las features de píxel son inservibles y vamos a necesitar HOG o embeddings de CNN sí o sí.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sample = df.groupby('compound_id').head(5).sample(min(400, len(df)), random_state=0)
X = np.stack([np.array(Image.open(ROOT / fp).convert('L').resize((32, 32))).flatten() for fp in sample['filepath']]).astype(float)
X = X / 255.0

pca = PCA(n_components=2, random_state=0).fit_transform(X)
tsne = TSNE(n_components=2, random_state=0, perplexity=min(30, max(5, len(X)//4))).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cat, color in zip(['inorganica', 'organica'], ['tab:blue', 'tab:orange']):
    mask = sample['category'].values == cat
    axes[0].scatter(pca[mask, 0], pca[mask, 1], label=cat, alpha=0.7, color=color, s=20)
    axes[1].scatter(tsne[mask, 0], tsne[mask, 1], label=cat, alpha=0.7, color=color, s=20)
axes[0].set_title('PCA (32×32 → 2D)'); axes[0].legend()
axes[1].set_title('t-SNE (32×32 → 2D)'); axes[1].legend()
plt.tight_layout(); plt.show()

## Pares confundibles

Para cada par de clases calculamos la SSIM (structural similarity index) entre sus *imágenes medias* y nos quedamos con los cinco pares más altos. La intuición: si dos clases tienen una media muy parecida, una red entrenada con pocas épocas las va a confundir. Esta lista nos sirve después para validar la matriz de confusión del notebook 03: si los errores se concentran en los pares predichos aquí, sabemos que el modelo está aprendiendo (los errores tienen sentido). Si no, hay que sospechar.

En la práctica, los pares que salen son los que cualquier alumno de bachillerato se confundiría: alcanos de longitud parecida, oxoácidos con el mismo átomo central y distinto número de oxígenos, sales con anión común y catión diferente.

In [ ]:
from skimage.metrics import structural_similarity as ssim

ids = df['compound_id'].drop_duplicates().sample(min(30, df['compound_id'].nunique()), random_state=0).tolist()
means = {}
for cid in ids:
    fps = df[df['compound_id'] == cid]['filepath'].head(5)
    arrs = [np.array(Image.open(ROOT / fp).convert('L').resize((64, 64))).astype(float) for fp in fps]
    means[cid] = np.mean(arrs, axis=0)

ids_l = list(means.keys())
S = np.zeros((len(ids_l), len(ids_l)))
for i, a in enumerate(ids_l):
    for j, b in enumerate(ids_l):
        S[i, j] = ssim(means[a], means[b], data_range=255)

pairs = []
for i in range(len(ids_l)):
    for j in range(i+1, len(ids_l)):
        pairs.append((ids_l[i], ids_l[j], S[i, j]))
pairs.sort(key=lambda x: -x[2])
print('Top-5 pares más similares (potencialmente confundibles):')
for a, b, s in pairs[:5]:
    print(f'  {a:20s} <-> {b:20s} SSIM={s:.3f}')

## Lo que aprendemos del EDA

Recapitulando lo que vamos a aplicar en los siguientes notebooks:

1. **Volumen suficiente, balanceo perfecto**. Con 300 imágenes por clase y 196 clases tenemos un dataset cómodo para una CNN, sin necesidad de oversampling agresivo. El `WeightedRandomSampler` lo dejamos puesto como red de seguridad.

2. **Dos modos visuales**. La proyección t-SNE confirma que el dataset tiene una dualidad estructura/texto. Para una red de visión convolucional esta heterogeneidad no es un problema, pero sí explica por qué los embeddings de ResNet18 funcionan tan bien en el ML clásico del notebook 02: las features convolucionales pre-entrenadas son lo bastante genéricas como para tratar ambos modos.

3. **Augmentación moderada**. Los pares confundibles que aparecen son sutiles. Aplicar augmentación demasiado agresiva (rotación grande, blur fuerte) destruiría la información que diferencia un alcano de su vecino. Por eso en `src/augmentation.py` mantenemos rotaciones de ±15°, ruido gaussiano suave y deformación elástica de baja amplitud.

4. **Métricas adecuadas**. Con 196 clases y un dataset balanceado, la accuracy global es informativa, pero no captura los errores localizados en pares confundibles. En el notebook 03 reportamos también macro-F1 (penaliza más las clases peor clasificadas) y matriz de confusión.